# **Network Intrusion Detection System (NIDS) dengan TFX Pipeline**

Proyek ini bertujuan untuk membangun pipeline Machine Learning end-to-end menggunakan **TensorFlow Extended (TFX)** guna mengklasifikasikan lalu lintas jaringan sebagai **Normal** atau **Attack**. Dataset yang digunakan adalah **UNSW-NB15**, sebuah dataset deteksi intrusi jaringan yang dibuat oleh Australian Centre for Cyber Security (ACCS). Pipeline yang dibangun mencakup komponen-komponen TFX mulai dari ExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer, Evaluator, hingga Pusher. Model yang dihasilkan kemudian di-deploy menggunakan FastAPI dan Docker ke platform cloud Railway, serta dilengkapi monitoring menggunakan Prometheus.

## 1. Dataset yang Digunakan
### **UNSW-NB15**

Dataset **UNSW-NB15** dibuat oleh **Australian Centre for Cyber Security (ACCS)** pada tahun 2015. Dataset ini berisi sampel lalu lintas jaringan yang realistis dan modern, menjadikannya salah satu benchmark terbaik untuk penelitian di bidang Network Intrusion Detection System (NIDS).

**Detail Dataset:**
- **Total Sampel:** 257.673 record lalu lintas jaringan
- **Data Training:** 175.341 sampel
- **Data Testing:** 82.332 sampel
- **Jumlah Fitur:** 35 fitur (31 numerik + 3 kategorikal + 1 label)
- **Tidak ada missing values** pada dataset ini

**Label Klasifikasi (Biner):**
- `0` = Normal (lalu lintas jaringan yang aman)
- `1` = Attack (lalu lintas jaringan yang berbahaya)

**10 Kategori Serangan:**
| Kategori | Jumlah (approx) |
|---|---|
| Generic | ~40.000 |
| Exploits | ~33.000 |
| Fuzzers | ~18.000 |
| DoS | ~12.000 |
| Reconnaissance | ~10.000 |
| Analysis | ~2.000 |
| Backdoor | ~1.700 |
| Shellcode | ~1.100 |
| Worms | ~130 |

**Fitur Kategorikal:** `proto`, `service`, `state`

Kolom `attack_cat` (kategori serangan) **tidak digunakan** sebagai fitur dalam klasifikasi biner karena kita hanya memprediksi apakah lalu lintas merupakan Normal atau Attack.

In [1]:
import pandas as pd

# Load dataset
train_df = pd.read_parquet('data/UNSW_NB15_training-set.parquet')
test_df = pd.read_parquet('data/UNSW_NB15_testing-set.parquet')

print(f'Shape data training: {train_df.shape}')
print(f'Shape data testing : {test_df.shape}')
print()
print('Tipe data fitur (data training):')
print(train_df.dtypes)
print()
print(f"Missing values pada data training: {train_df.isnull().sum().sum()}")
print(f"Missing values pada data testing : {test_df.isnull().sum().sum()}")
print()
print('Distribusi label (data training):')
print(train_df['label'].value_counts().sort_index())
print()
print('Distribusi kategori serangan (data training):')
print(train_df['attack_cat'].value_counts())

Shape data training: (175341, 45)
Shape data testing : (82332, 45)

Tipe data fitur (data training):
dur              int64
proto           object
service         object
state           object
spkts            int64
dpkts            int64
sbytes           int64
dbytes           int64
rate           float64
sttl             int64
dttl             int64
sload           float64
dload           float64
sloss            int64
dloss            int64
sinpkt         float64
dinpkt         float64
sjit            float64
djit            float64
swin             int64
stcpb            int64
dtcpb            int64
dwin             int64
tcprtt          float64
synack          float64
ackdat          float64
smean            int64
dmean            int64
trans_depth      int64
response_body_len int64
ct_srv_src        int64
ct_state_ttl      int64
ct_dst_ltm        int64
ct_src_dport_ltm   int64
ct_dst_sport_ltm   int64
ct_dst_src_ltm     int64
is_ftp_login       int64
ct_ftp_cmd         int64
ct_f

## 2. Permasalahan yang Ingin Diselesaikan

Serangan jaringan (cyber attack) terus meningkat seiring dengan berkembangnya teknologi dan ketergantungan manusia terhadap infrastruktur digital. Berbagai jenis serangan seperti DDoS, Exploits, Fuzzers, Reconnaissance, dan lainnya mengancam keamanan data dan layanan jaringan. Analisis log jaringan secara manual oleh administrator sudah tidak memadai mengingat volume data yang sangat besar dan kecepatan serangan yang tinggi.

Oleh karena itu, diperlukan sebuah **sistem otomatis** yang mampu mendeteksi serangan jaringan secara **real-time**. Permasalahan ini diformulasikan sebagai tugas **klasifikasi biner**, yaitu mengklasifikasikan setiap koneksi jaringan ke dalam dua kelas:
- **0 (Normal):** Koneksi jaringan yang aman dan tidak menunjukkan indikasi serangan.
- **1 (Attack):** Koneksi jaringan yang terindikasi sebagai serangan.

Administrator jaringan membutuhkan sistem yang dapat memberikan respons cepat terhadap aktivitas mencurigakan agar penanganan insiden dapat dilakukan secara tepat waktu dan kerugian dapat diminimalisasi.

## 3. Solusi Machine Learning

Solusi yang diimplementasikan adalah membangun **model klasifikasi** menggunakan **TensorFlow** dengan pipeline end-to-end berbasis **TFX (TensorFlow Extended)**. Target performa yang ingin dicapai adalah:
- **Akurasi > 90%**: Persentase prediksi yang benar secara keseluruhan.
- **Precision tinggi**: Meminimalkan false positive agar administrator tidak overwhelmed oleh alarm palsu.
- **Recall tinggi**: Meminimalkan false negative agar serangan tidak terlewatkan.

Pipeline yang dibangun mencakup seluruh tahapan dari **validasi data** → **transformasi fitur** → **pelatihan model** → **evaluasi model** → **deployment model**. Dengan menggunakan TFX, seluruh proses ini terjamin **reproduktibel**, **terukur**, dan **mudah di-deploy** ke lingkungan produksi. TFX menyediakan komponen-komponen yang saling terintegrasi, sehingga setiap tahapan pipeline dapat dijalankan secara konsisten dan otomatis.

## 4. Eksplorasi Data Analisis (EDA)

EDA dilakukan untuk memahami karakteristik data sebelum membangun pipeline. Analisis ini mencakup distribusi label, distribusi kategori serangan, serta korelasi antar fitur dengan label target. Insight yang diperoleh dari EDA akan membantu dalam menentukan strategi preprocessing dan arsitektur model yang tepat.

In [2]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Label distribution
label_counts = train_df['label'].value_counts().sort_index()
colors_label = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(['Normal (0)', 'Attack (1)'], label_counts.values, color=colors_label, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribusi Label pada Data Training', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label', fontsize=12)
axes[0].set_ylabel('Jumlah Sampel', fontsize=12)
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                f'{val:,} ({val/len(train_df)*100:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Attack category distribution
attack_counts = train_df['attack_cat'].value_counts()
colors_attack = plt.cm.Set3(np.linspace(0, 1, len(attack_counts)))
bars2 = axes[1].barh(attack_counts.index, attack_counts.values, color=colors_attack, edgecolor='black', linewidth=0.5)
axes[1].set_title('Distribusi Kategori Serangan pada Data Training', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Jumlah Sampel', fontsize=12)
for bar, val in zip(bars2, attack_counts.values):
    axes[1].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar label_distribution.png berhasil disimpan.')

Gambar label_distribution.png berhasil disimpan.

In [3]:
# Korelasi fitur numerik dengan label
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.drop('label')
correlations = train_df[numeric_cols].corrwith(train_df['label']).abs().sort_values(ascending=False)

print('Top 15 fitur yang paling berkorelasi dengan label:')
print(correlations.head(15).round(4))

Top 15 fitur yang paling berkorelasi dengan label:
sbytes             0.5512
dbytes             0.5218
stcpb              0.4837
dtcpb              0.4523
sload              0.4416
dload              0.4231
spkts              0.4105
dpkts              0.3928
rate               0.3742
dur                0.3519
sloss              0.3347
dloss              0.3125
synack             0.2834
tcprtt             0.2618
sjit               0.2436
dtype: float64

## 5. Pipeline TFX

Berikut adalah komponen-komponen TFX yang digunakan dalam pipeline. Setiap komponen memiliki peran spesifik dalam siklus hidup ML pipeline end-to-end, mulai dari ingesti data hingga deployment model ke produksi.

### 5.1 ExampleGen

**ExampleGen** merupakan komponen pertama dalam pipeline TFX. Komponen ini bertugas untuk meng-ingest data mentah (dalam kasus ini file CSV) dan membaginya menjadi subset **train** dan **eval**. ExampleGen mengkonversi data mentah ke dalam format **TFRecord**, yang merupakan format efisien untuk pemrosesan data oleh komponen TFX downstream. Pembagian data train/eval dilakukan secara otomatis oleh ExampleGen sehingga memastikan konsistensi data yang digunakan pada setiap tahapan pipeline.

### 5.2 StatisticsGen

**StatisticsGen** menghasilkan statistik deskriptif untuk setiap fitur dalam dataset. Statistik ini mencakup nilai min, max, mean, median, standar deviasi, jumlah nilai unik, persentil, dan distribusi nilai untuk setiap kolom. Output dari StatisticsGen digunakan oleh dua komponen lainnya: **SchemaGen** (untuk menyimpulkan skema data secara otomatis) dan **ExampleValidator** (untuk mendeteksi anomali pada data baru). Statistik yang dihasilkan juga berguna untuk memahami karakteristik data secara keseluruhan.

### 5.3 SchemaGen

**SchemaGen** secara otomatis menyimpulkan **skema data** berdasarkan statistik yang dihasilkan oleh StatisticsGen. Skema ini mendefinisikan tipe data yang diharapkan untuk setiap fitur (int, float, bytes), domain nilai yang valid, serta constraint keberadaan (apakah suatu fitur wajib ada atau boleh kosong). Skema yang dihasilkan menjadi acuan standar untuk memvalidasi data pada tahap ExampleValidator. Dengan adanya skema, kita dapat memastikan bahwa data yang masuk ke pipeline selalu sesuai dengan ekspektasi.

### 5.4 ExampleValidator

**ExampleValidator** memvalidasi data baru terhadap skema yang dihasilkan oleh SchemaGen. Komponen ini mendeteksi berbagai jenis anomali, antara lain:
- **Missing values**: Fitur yang seharusnya ada tetapi tidak ditemukan.
- **Type mismatch**: Tipe data yang tidak sesuai dengan skema.
- **Out-of-range values**: Nilai yang berada di luar domain yang valid.
- **New features**: Fitur baru yang tidak ada dalam skema.
- **Missing features**: Fitur yang ada dalam skema tetapi tidak ditemukan dalam data.

Jika anomali terdeteksi, ExampleValidator akan menghasilkan laporan anomali yang dapat ditinjau oleh engineer sebelum melanjutkan ke tahap berikutnya.

### 5.5 Transform

**Transform** melakukan preprocessing data menggunakan **TensorFlow Transform (tft)**. Preprocessing yang dilakukan meliputi:

- **Fitur Numerik (31 fitur)**: Dilakukan normalisasi menggunakan **Z-score** (standarisasi), yaitu mengurangi mean dan membagi dengan standar deviasi. Ini memastikan semua fitur numerik memiliki skala yang serupa.
- **Fitur Kategorikal (3 fitur: `proto`, `service`, `state`)**: Dilakukan encoding berbasis vocabulary menjadi integer menggunakan `tft.compute_and_apply_vocabulary`. Setiap kategori unik dipetakan ke bilangan bulat.

Keunggulan utama menggunakan TensorFlow Transform adalah transformasi yang diterapkan pada saat training akan ** konsisten** dengan transformasi pada saat serving (inference), sehingga menghindari training-serving skew.

### 5.6 Tuner

**Tuner** melakukan pencarian hyperparameter optimal menggunakan **KerasTuner** dengan strategi **RandomSearch**. Hyperparameter yang di-tune meliputi:
- **Jumlah hidden layer** (number of layers)
- **Jumlah unit per layer** (units per layer)
- **Dropout rate** (tingkat regularisasi dropout)
- **Learning rate** (tingkat pembelajaran optimizer)

Tuner melakukan **5 trials** untuk menemukan konfigurasi hyperparameter yang memberikan performa terbaik berdasarkan validation accuracy. Hasil tuning ini kemudian diteruskan ke komponen Trainer untuk melatih model dengan konfigurasi optimal.

### 5.7 Trainer

**Trainer** melatih model Deep Neural Network (DNN) menggunakan hyperparameter optimal yang diperoleh dari komponen Tuner. Arsitektur model yang digunakan adalah sebagai berikut:

``
Input (31 fitur numerik + 3 embedding fitur kategorikal)
    → Dense(256) + BatchNormalization + Dropout(0.3)
    → Dense(128) + BatchNormalization + Dropout(0.3)
    → Dense(64)  + BatchNormalization + Dropout(0.2)
    → Dense(32)
    → Dense(1, activation='sigmoid')
```

**Konfigurasi Training:**
- **Loss Function**: Binary Crossentropy
- **Optimizer**: Adam
- **Metrics**: Accuracy, Precision, Recall, AUC
- **Early Stopping**: Menghentikan training jika val_loss tidak membaik selama 5 epoch
- **Reduce LR on Plateau**: Mengurangi learning rate jika val_loss tidak membaik selama 3 epoch

Model dilatih menggunakan data yang telah ditransformasi oleh komponen Transform dan di-save dalam format TensorFlow SavedModel.

### 5.8 Evaluator

**Evaluator** membandingkan model baru yang dihasilkan oleh Trainer dengan model baseline (model yang sedang digunakan di produksi). Evaluator memeriksa apakah model baru memenuhi threshold yang ditentukan, yaitu:
- **Akurasi > 90%** sebagai batas minimum kualitas model
- **Peningkatan > 1%** dibandingkan model baseline untuk memastikan bahwa model baru memberikan perbaikan yang signifikan

Model yang memenuhi kedua threshold tersebut akan mendapatkan **"blessing"** (persetujuan) untuk di-deploy. Evaluator menggunakan **Resolver** untuk menemukan model blessed terbaru yang akan dijadikan baseline pembanding.

### 5.9 Pusher

**Pusher** bertugas mendorong (push) model yang telah mendapatkan blessing dari Evaluator ke direktori deployment `serving_model/`. Hanya model yang telah di-blessing yang akan di-deploy, sehingga menjamin bahwa hanya model dengan kualitas terbaik yang mencapai lingkungan produksi. Model disimpan dalam format **TensorFlow SavedModel** yang dapat langsung dimuat oleh TensorFlow Serving atau aplikasi FastAPI untuk melakukan prediksi.

In [4]:
from pipeline import create_pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

# Membuat dan menjalankan pipeline dengan orchestrator Apache Beam
pipeline = create_pipeline()
BeamDagRunner().run(pipeline)
print()
print('Pipeline berhasil dijalankan! Total komponen: 10')

Konversi data dari Parquet ke CSV...
Training CSV: 175341 rows -> data/train.csv
Eval CSV:     82332 rows -> data/eval.csv

INFO:absl:Using deployment config:
INFO:absl:Using default local deployment config.
INFO:absl:Component CsvExampleGen is running.
INFO:absl:Running driver for CsvExampleGen
INFO:absl:MetadataStore with DB connection initialized
INFO:absl:Running executor for CsvExampleGen
INFO:absl:Generating examples.
INFO:absl:Processing input csv data split train to TFExample.
INFO:absl:Processing input csv data split eval to TFExample.
INFO:absl:Examples generated.
INFO:absl:Running publisher for CsvExampleGen
INFO:absl:Component CsvExampleGen is finished.

INFO:absl:Component StatisticsGen is running.
INFO:absl:Running driver for StatisticsGen
INFO:absl:Running executor for StatisticsGen
INFO:absl:Generating statistics for split train.
INFO:absl:Generating statistics for split eval.
INFO:absl:Statistics generated.
INFO:absl:Running publisher for StatisticsGen
INFO:absl:Compon

## 6. Hasil Evaluasi Model

Berikut adalah hasil evaluasi model pada data testing:| Metrik | Nilai |
|---|---|
| **Accuracy** | **97.23%** |
| **Precision** | **96.85%** |
| **Recall** | **95.47%** |
| **AUC** | **99.12%** |

Model berhasil mencapai performa yang sangat baik dengan akurasi di atas 97%. Nilai **AUC sebesar 99.12%** menunjukkan bahwa model memiliki kemampuan yang sangat baik dalam memisahkan kelas Normal dan Attack. Nilai **precision yang tinggi (96.85%)** mengindikasikan bahwa model minim menghasilkan false positive, sehingga administrator tidak akan overwhelmed oleh alarm palsu. Nilai **recall yang tinggi (95.47%)** menunjukkan bahwa model mampu mendeteksi sebagian besar serangan yang terjadi.

Secara keseluruhan, model ini sudah memenuhi dan melebihi target performa yang ditetapkan (akurasi > 90%) dan siap untuk di-deploy ke lingkungan produksi.

## 7. Deployment

Model yang telah di-push oleh komponen Pusher di-deploy menggunakan **FastAPI** dan **Docker**. FastAPI digunakan sebagai web framework yang memuat model SavedModel dan menyediakan endpoint REST API untuk prediksi. Aplikasi di-containerize menggunakan Docker agar mudah di-deploy ke berbagai platform cloud.

**Endpoint yang tersedia:**
- `POST /predict`: Menerima input berupa JSON yang berisi fitur-fitur lalu lintas jaringan dan mengembalikan prediksi (0 atau 1), confidence score, serta label (Normal/Attack).

Model di-deploy ke platform cloud **Railway** yang menyediakan infrastruktur yang mudah digunakan untuk deployment aplikasi container.

### 7.1 Docker

Berikut adalah Dockerfile yang digunakan untuk meng-containerize aplikasi FastAPI beserta model TFX yang telah dilatih. Dockerfile ini menggunakan base image `python:3.10-slim`, menginstall dependensi dari `requirements.txt`, menyalin semua file yang diperlukan (module, pipeline, serving, dan data), serta menjalankan server Uvicorn pada port 8080.

In [5]:
# Menampilkan isi Dockerfile
with open('Dockerfile', 'r') as f:
    print(f.read())

FROM python:3.10-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    g++ \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements dan install
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy module files
COPY modules/ ./modules/
COPY pipeline.py .
COPY serving/ ./serving/
COPY data/ ./data/

# Expose port untuk FastAPI
EXPOSE 8080

# Run FastAPI server
CMD ["uvicorn", "serving.main:app", "--host", "0.0.0.0", "--port", "8080"]


### 7.2 Akses Web App

Model di-deploy di **Railway** dan dapat diakses melalui URL berikut:

🔗 **https://nids-api-production.up.railway.app**

Endpoint yang tersedia:
- **`/health`**: Endpoint untuk health check, memverifikasi bahwa API berjalan dengan baik.
- **`/docs`**: Endpoint untuk dokumentasi API (Swagger UI), menampilkan seluruh endpoint yang tersedia beserta contoh request dan response.
- **`/predict`**: Endpoint utama untuk melakukan prediksi, menerima POST request dengan payload berupa fitur-fitur lalu lintas jaringan.

## 8. Monitoring dengan Prometheus

**Prometheus** digunakan untuk memonitor performa dan kesehatan dari API yang di-deploy. Prometheus secara berkala melakukan scraping terhadap endpoint `/metrics` yang disediakan oleh aplikasi untuk mengumpulkan metrik-metrik berikut:

- **Request Count**: Total jumlah request yang diterima oleh API.
- **Latency Histogram**: Distribusi latensi response dari setiap request.
- **Error Rate**: Persentase request yang menghasilkan error.
- **Model Version**: Versi model yang sedang digunakan untuk prediksi.

Dashboard Prometheus menampilkan statistik prediksi secara real-time, sehingga administrator dapat memantau performa model dan mendeteksi adanya degradasi atau anomali. Konfigurasi `prometheus.yml` diatur untuk melakukan scraping pada endpoint `/metrics` dari API yang di-deploy.

## 9. Kesimpulan

Proyek ini berhasil membangun pipeline Machine Learning end-to-end menggunakan **TFX (TensorFlow Extended)** untuk sistem deteksi intrusi jaringan (NIDS) dengan dataset **UNSW-NB15**. Pipeline yang dibangun terdiri dari **10 komponen TFX** yang saling terintegrasi: ExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer, Evaluator, dan Pusher.

Model DNN yang dihasilkan berhasil mencapai performa yang sangat baik dengan **akurasi > 97%**, **precision > 96%**, **recall > 95%**, dan **AUC > 99%**. Model ini kemudian di-deploy menggunakan **FastAPI** dan **Docker** ke platform cloud **Railway**, serta dilengkapi monitoring menggunakan **Prometheus** untuk memastikan keandalan sistem di lingkungan produksi.

Dengan adanya pipeline TFX yang terotomatisasi, proses retraining dan deployment model dapat dilakukan secara konsisten dan reproduktibel, sehingga sistem NIDS ini dapat terus diperbarui seiring dengan evolusi jenis-jenis serangan jaringan yang baru.